In [1]:
from Declare4Py.ProcessModels.DeclareModel import DeclareModel
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner
from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessModels.DeclareModel import DeclareModelTemplate
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareResultsBrowser import MPDeclareResultsBrowser


import pickle, os
import pandas as pd
import numpy as np


In [2]:
event_log_name = "large"
log_path = f"D:\\LTNcoder\\.out\\eventlogs\\{event_log_name}-0.3-1.xes"

event_log = D4PyEventLog(case_name="case:concept:name")
event_log.parse_xes_log(log_path)

c:\Users\devas\anaconda3\envs\ltn\lib\site-packages\pm4py\util\dt_parsing\parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/5000 [00:00<?, ?it/s]

In [3]:
# Save the conformance checking results to disk to avoid recalculating
def save_conformance_results(conf_check_res, filename=f'{event_log_name}_5000_conformance_results.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(conf_check_res, f)
    print(f"Conformance checking results saved to {filename}")

# Load the conformance checking results from disk
def load_conformance_results(filename=f'{event_log_name}_5000_conformance_results.pkl'):
    try:
        with open(filename, 'rb') as f:
            conf_check_res = pickle.load(f)
        print(f"Conformance checking results loaded from {filename}")
        return conf_check_res
    except FileNotFoundError:
        print(f"File {filename} not found. Run conformance checking first.")
        return None

In [4]:
if not os.path.exists(f"{event_log_name}-0.3-1.decl"):
    print(f"File {event_log_name}-0.3-1.decl does not exist, running discovery...")
    discovery = DeclareMiner(log=event_log, consider_vacuity=False, min_support=0.05, itemsets_support=0.05, max_declare_cardinality=1)
    declare_model: DeclareModel = discovery.run()
    print(f"Total constraints discovered: {len(declare_model.serialized_constraints)}")
    model_constraints = declare_model.get_decl_model_constraints()
    declare_model.to_file(f"{event_log_name}-0.3-1.decl")

File large-0.3-1.decl does not exist, running discovery...
Computing discovery ...
Total constraints discovered: 4627


In [5]:
if not os.path.exists(f'{event_log_name}_5000_conformance_results.pkl') and event_log is not None and declare_model is not None:
    print(f"File {event_log_name}_5000_conformance_results.pkl does not exist, running conformance checking...")
    basic_checker = MPDeclareAnalyzer(log=event_log, declare_model=declare_model, consider_vacuity=False)
    conf_check_res: MPDeclareResultsBrowser = basic_checker.run()
    save_conformance_results(conf_check_res)
else:
    print(f"Loading conformance checking results from {event_log_name}_5000_conformance_results.pkl")
    conf_check_res = load_conformance_results()
conf_check_df =  conf_check_res.get_metric(metric="state")


File large_5000_conformance_results.pkl does not exist, running conformance checking...
Conformance checking results saved to large_5000_conformance_results.pkl


In [6]:
activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)
satisfied_df = conf_check_res.get_metric(metric="state").fillna(0)

# Support = how often constraint is satisfied
support = satisfied_df.sum(axis=0) / len(satisfied_df)

# Activation rate = how often constraint is activated
activated_counts = activated_df.sum(axis=0)
activation_rate = activated_counts / len(activated_df)

# Satisfied counts
satisfied_counts = satisfied_df.sum(axis=0)

# Confidence = P(satisfied | activated), safe division
confidence = np.where(
    activated_counts != 0,
    satisfied_counts / activated_counts,
    0
)
confidence = pd.Series(confidence, index=activated_counts.index)

# Combine into a DataFrame
metrics_df = pd.DataFrame({
    'support': support,
    'confidence': confidence,
    # 'activation_rate': activation_rate
})
# metrics_df


C:\Users\devas\AppData\Local\Temp\ipykernel_72716\1810640346.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)


In [7]:
# filtered_metrics_df is metrics_df but with rows with support less than 0.2 and in descending order of confidence and no "not" in the constraint name
filtered_metrics_df = metrics_df[~metrics_df.index.str.contains("Not") & (metrics_df['support'] <= 0.2) & (metrics_df['confidence'] >= 0.7)].sort_values(by='confidence', ascending=False)
# filtered_metrics_df = metrics_df[metrics_df['support'] <= 0.2].sort_values(by='confidence', ascending=False)
print("Filtered Metrics DataFrame:")
display(filtered_metrics_df)

Filtered Metrics DataFrame:


,support,confidence
"Responded Existence[Activity U, Activity B] | |",0.0792,0.992481
"Response[Activity AE, Activity B] | |",0.1198,0.991722
"Responded Existence[Activity AE, Activity B] | |",0.1198,0.991722
"Responded Existence[Activity AD, Activity B] | |",0.1298,0.990840
"Response[Activity AD, Activity B] | |",0.1298,0.990840
...,...,...
"Response[Activity R, Activity U] | |",0.0752,0.917073
"Chain Precedence[Activity R, Activity S] | |",0.0750,0.916870
"Alternate Response[Activity Q, Activity U] | |",0.0748,0.916667
"Chain Response[Activity R, Activity S] | |",0.0746,0.909756


In [8]:
raise KeyboardInterrupt("\nStopping execution after displaying filtered metrics DataFrame.\nChoose your favourite constraints from the filtered_metrics_df DataFrame and add them to the interesting_constraints list in the ipynb.\nThen run the cells below again to see the results of the selected constraints.")

KeyboardInterrupt: 
Stopping execution after displaying filtered metrics DataFrame.
Choose your favourite constraints from the filtered_metrics_df DataFrame and add them to the interesting_constraints list in the ipynb.
Then run the cells below again to see the results of the selected constraints.

# Constraints with low support and high confidence
1. Responded Existence[Activity U, Activity B] | |	0.0792	0.9924812030075187

In [13]:
interesting_constraints = [
    # "Response[Approve PO 2, Release PO] | |",
    "Responded Existence[Activity U, Activity B] | |"
    ]

In [14]:
state_df = conf_check_res.get_metric("state")[interesting_constraints]
non_zero_counts = state_df.ne(0).sum(axis=0)
print(non_zero_counts)
non_zero_rows = state_df.index[state_df[interesting_constraints[0]] != 0].tolist()
print(non_zero_rows)
print(len(non_zero_rows))
with open(f'{event_log_name}_ltn_rows.pkl', 'wb') as f:
    pickle.dump(non_zero_rows, f)

Responded Existence[Activity U, Activity B] | |    396
dtype: int64
[26, 29, 38, 49, 50, 65, 73, 74, 80, 81, 82, 88, 92, 112, 113, 115, 129, 179, 188, 193, 210, 220, 234, 239, 246, 249, 250, 251, 289, 339, 370, 375, 387, 391, 420, 441, 449, 451, 454, 463, 468, 473, 489, 492, 497, 499, 503, 510, 512, 521, 529, 541, 545, 547, 549, 551, 556, 571, 584, 589, 597, 614, 633, 638, 654, 676, 699, 707, 721, 727, 729, 734, 743, 764, 781, 815, 823, 824, 831, 835, 837, 841, 874, 885, 927, 929, 933, 960, 983, 1003, 1027, 1029, 1055, 1092, 1130, 1132, 1138, 1140, 1144, 1150, 1163, 1190, 1209, 1210, 1232, 1236, 1254, 1270, 1290, 1298, 1313, 1321, 1337, 1342, 1357, 1362, 1395, 1400, 1401, 1406, 1413, 1417, 1427, 1428, 1436, 1461, 1469, 1485, 1498, 1512, 1526, 1528, 1561, 1562, 1597, 1603, 1618, 1655, 1665, 1669, 1681, 1721, 1722, 1726, 1737, 1742, 1752, 1801, 1803, 1811, 1815, 1824, 1834, 1837, 1841, 1866, 1880, 1882, 1883, 1895, 1913, 1926, 1933, 1965, 1967, 2018, 2040, 2046, 2047, 2051, 2061, 2122, 2

In [15]:
print("END")

END


In [12]:
# summary_df = conf_check_df.apply(lambda col: col.value_counts()).fillna(0).astype(int)
# summary_df = summary_df.reindex([0, 1])
# summary_df = summary_df / len(conf_check_df)
# summary_df = summary_df.T
# summary_df = summary_df.sort_values(by=1, ascending=False)
# display(summary_df)